# E2 — TAPE feature × GNN matrix on Goodreads-Children

**Per-seed runner.** Set `SEED` in the config cell below, run the whole notebook end-to-end. Each seed produces independent artifacts under `results/runs/goodreads_children/seed_<S>/`. Re-run the notebook with `SEED = 1, 2, 3` to fill the rest of the table — earlier seeds are not touched.

**Stages:**
1. Mount Drive + clone repo + install deps
2. Build Goodreads-Children TAG (one-time; cached on Drive)
3. Generate LLM explanations + top-k preds (one-time; cached on Drive)
4. **Per seed:** fine-tune DeBERTa twice (TA on raw text, E on LLM explanations)
5. **Per seed:** run all 5 feature × 2 GNN runs
6. Dump per-seed result JSONs to `results/runs/.../seed_<S>/`

Artifacts that survive across seeds (only built once):
- `new_dataset/data/goodreads_children/{graph.pt, node_texts.jsonl, labels.txt}`
- `TAPE/gpt_responses/goodreads_children/<i>.json` (76k files)
- `TAPE/gpt_preds/goodreads_children.csv`

Artifacts produced **per seed**:
- `TAPE/prt_lm/goodreads_children/microsoft/deberta-base-seed{S}.{ckpt,emb,pred}` (TA)
- `TAPE/prt_lm/goodreads_children2/microsoft/deberta-base-seed{S}.{ckpt,emb,pred}` (E)
- `results/runs/goodreads_children/seed_{S}/{feat}_{gnn}.json` (10 files: 5 feats × 2 GNNs)

## 0. Configuration

In [ ]:
# ===================== EDIT THIS PER RUN =====================
SEED = 0                     # the only knob you flip between runs (0, 1, 2, 3)

# Subsampling: set to a small int (e.g. 1500) for a fast end-to-end shake-out;
# set to None for the full ~76k-node run. The DATASET slug below changes with
# LIMIT_NODES so subsample artifacts and full-run artifacts don't collide on Drive.
LIMIT_NODES = 1500           # set to None when ready for the full run

BASE_DATASET = 'goodreads_children'
DATASET = BASE_DATASET if LIMIT_NODES is None else f'{BASE_DATASET}_n{LIMIT_NODES}'
LM_MODEL = 'microsoft/deberta-base'
LLM_MODEL = 'gpt-4o-mini'
GNNS = ['GCN', 'SAGE']
SINGLE_FEATURES = ['ogb', 'TA', 'P', 'E']
ENSEMBLE_FEATURE = 'TA_P_E'

# Drive layout: everything lives under /content/drive/MyDrive/graphml-project/
DRIVE_ROOT  = '/content/drive/MyDrive/graphml-project'
REPO_URL    = 'https://github.com/GeorgeZhang20/graphml-project.git'
REPO_BRANCH = 'main'

# Auto-push small artifacts (CSV summaries + figures) back to GitHub at the end
# of every seed so they show up locally too. Set False to skip.
AUTO_PUSH      = True
GIT_USER_EMAIL = 'qz51@rice.edu'
GIT_USER_NAME  = 'George Zhang (Colab)'
# =============================================================

print(f'Configured  SEED={SEED}  DATASET={DATASET}  LIMIT_NODES={LIMIT_NODES}')


## 1. Drive + clone + install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.chdir(DRIVE_ROOT)

# Symlink so repo paths work without copying. The first run clones into Drive;
# subsequent runs just `git pull`.
if not os.path.isdir(os.path.join(DRIVE_ROOT, '.git')):
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, DRIVE_ROOT])
else:
    subprocess.check_call(['git', 'pull', '--ff-only'])

!pwd && ls

In [ ]:
# Colab T4 has CUDA 12. Install GPU-flavored torch + PyG and the rest from requirements.txt.
!pip install -q --upgrade pip
!pip install -q torch==2.2.2 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch_geometric ogb 'transformers>=4.40,<4.50' 'accelerate>=0.30' \
                'numpy<2' scikit-learn pandas tqdm yacs sentence-transformers openai
import torch; print('CUDA:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Build the TAG (one-time; cached on Drive)

In [ ]:
from pathlib import Path
import subprocess, sys

data_dir = Path(DRIVE_ROOT) / 'new_dataset' / 'data' / DATASET
raw_csv  = Path(DRIVE_ROOT) / 'new_dataset' / 'raw' / 'Children.csv'   # shared across runs
expected = ['graph.pt', 'node_texts.jsonl', 'labels.txt']
if all((data_dir / f).exists() for f in expected):
    print(f'[skip] {data_dir} already populated:')
    !ls -lh {data_dir}
else:
    cmd = [sys.executable, 'new_dataset/prep/build_goodreads_children.py',
           '--csv', str(raw_csv),
           '--out_dir', str(data_dir)]
    if LIMIT_NODES is not None:
        cmd += ['--limit_nodes', str(LIMIT_NODES)]
    print('+', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print('---STDOUT---')
    print(proc.stdout)
    print('---STDERR---')
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f'build script exited with code {proc.returncode}; see stderr above')


## 3. Generate LLM explanations + top-k preds (one-time; cached on Drive)

`generate.py` is **resumable** — re-running it skips nodes whose JSON already exists. Cost on GPT-4o-mini for ~76k Goodreads-Children nodes ≈ \$15-30 (verify with a 100-node dry-run first).

In [ ]:
import os
from getpass import getpass
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

# Dry-run smoke check: 5 nodes, no real API calls. Verifies file plumbing only.
!python llm_explanations/generate.py \
    --dataset {DATASET} \
    --node_texts new_dataset/data/{DATASET}/node_texts.jsonl \
    --labels    new_dataset/data/{DATASET}/labels.txt \
    --responses_dir TAPE/gpt_responses/{DATASET} \
    --preds_csv TAPE/gpt_preds/{DATASET}.csv \
    --max_nodes 5 --dry_run


In [ ]:
# Cost-estimate: real call against 100 nodes. With gpt-4o-mini this is ~$0.04.
# After it finishes, eyeball one of the JSONs to confirm the 'explanation' looks
# coherent before going to the full run.
!python llm_explanations/generate.py \
    --dataset {DATASET} \
    --node_texts new_dataset/data/{DATASET}/node_texts.jsonl \
    --labels    new_dataset/data/{DATASET}/labels.txt \
    --responses_dir TAPE/gpt_responses/{DATASET} \
    --preds_csv TAPE/gpt_preds/{DATASET}.csv \
    --model {LLM_MODEL} \
    --max_nodes 100


In [ ]:
# Real full run. Resumable — interrupt + re-run picks up where it left off.
# Cost rough estimate: 76k nodes * gpt-4o-mini ≈ $15-30. Subsample (LIMIT_NODES=1500) ≈ $0.30.
!python llm_explanations/generate.py \
    --dataset {DATASET} \
    --node_texts new_dataset/data/{DATASET}/node_texts.jsonl \
    --labels    new_dataset/data/{DATASET}/labels.txt \
    --responses_dir TAPE/gpt_responses/{DATASET} \
    --preds_csv TAPE/gpt_preds/{DATASET}.csv \
    --model {LLM_MODEL}


## 4. LM fine-tunes (per seed — TA + E)

Fine-tunes DeBERTa-base twice for the current `SEED`. Each run takes ~1.5h on a T4.
Skips if the .emb file is already on Drive.

In [ ]:
import os
lm_safe = LM_MODEL.replace('/', '_')
ta_emb = f'TAPE/prt_lm/{DATASET}/{LM_MODEL}-seed{SEED}.emb'
e_emb  = f'TAPE/prt_lm/{DATASET}2/{LM_MODEL}-seed{SEED}.emb'

os.environ['WANDB_DISABLED'] = 'True'
os.environ['TOKENIZERS_PARALLELISM'] = 'False'

%cd {DRIVE_ROOT}/TAPE

if not os.path.exists(ta_emb):
    !python -m core.trainLM dataset {DATASET} seed {SEED}
else:
    print(f'[skip TA] {ta_emb} already exists')

if not os.path.exists(e_emb):
    !python -m core.trainLM dataset {DATASET} seed {SEED} lm.train.use_gpt True
else:
    print(f'[skip E] {e_emb} already exists')

## 5. GNN sweep for this seed (5 features × 2 GNNs = 10 runs)

Each run dumps its test/val acc to `results/runs/<DATASET>/seed_<S>/<feat>_<gnn>.json`.

In [ ]:
import json, os, re, subprocess, time
from pathlib import Path

results_dir = Path(DRIVE_ROOT) / 'results' / 'runs' / DATASET / f'seed_{SEED}'
results_dir.mkdir(parents=True, exist_ok=True)

def parse_acc_from_log(log: str):
    """Pull the last 'ValAcc: X.XXXX, TestAcc: Y.YYYY' line that gnn_trainer.py prints."""
    matches = re.findall(r'ValAcc: ([0-9.]+), TestAcc: ([0-9.]+)', log)
    if not matches:
        return None, None
    return float(matches[-1][0]), float(matches[-1][1])

def run_one(feature_type: str, gnn: str):
    out_path = results_dir / f'{feature_type}_{gnn}.json'
    if out_path.exists():
        print(f'[skip] {out_path} already exists')
        return
    entry = 'core.trainEnsemble' if feature_type == ENSEMBLE_FEATURE else 'core.trainGNN'
    cmd = ['python', '-m', entry,
           'dataset', DATASET,
           'gnn.model.name', gnn,
           'gnn.train.feature_type', feature_type,
           'seed', str(SEED)]
    print('+', ' '.join(cmd))
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    log = proc.stdout + '\n' + proc.stderr
    val_acc, test_acc = parse_acc_from_log(log)
    record = {
        'dataset': DATASET, 'seed': SEED, 'gnn': gnn,
        'feature': feature_type, 'val_acc': val_acc, 'test_acc': test_acc,
        'wall_seconds': time.time() - t0,
        'returncode': proc.returncode,
    }
    with open(out_path, 'w') as f:
        json.dump(record, f, indent=2)
    # also dump the log next to it for debugging
    with open(out_path.with_suffix('.log'), 'w') as f:
        f.write(log)
    print(f'  -> val={val_acc} test={test_acc} ({record["wall_seconds"]:.0f}s)')

%cd {DRIVE_ROOT}/TAPE

for feat in SINGLE_FEATURES + [ENSEMBLE_FEATURE]:
    for gnn in GNNS:
        run_one(feat, gnn)

print('\n[done seed', SEED, ']')

## 6. Inspect what we just produced

In [ ]:
%cd {DRIVE_ROOT}
!ls results/runs/{DATASET}/seed_{SEED}/
!python scripts/aggregate_results.py --dataset {DATASET}

## 7. Push artifacts back to GitHub (so they show up locally too)

Pushes the small summary CSVs + figures (NOT the per-run JSONs, NOT the .emb/.ckpt — those are gitignored). Needs a GitHub Personal Access Token with `repo` scope. Generate one at https://github.com/settings/tokens and paste when prompted; it gets stored only in this Colab session.

In [ ]:
import os, subprocess
from getpass import getpass

if AUTO_PUSH:
    %cd {DRIVE_ROOT}
    if not os.environ.get('GITHUB_TOKEN'):
        os.environ['GITHUB_TOKEN'] = getpass('GITHUB_TOKEN (PAT with `repo` scope): ')
    # Configure git identity for this Colab session (one-time per VM)
    subprocess.run(['git', 'config', 'user.email', GIT_USER_EMAIL], check=True)
    subprocess.run(['git', 'config', 'user.name',  GIT_USER_NAME],  check=True)
    # Use token-bearing remote URL only for this push
    token_url = REPO_URL.replace('https://', f'https://x-access-token:{os.environ["GITHUB_TOKEN"]}@')
    # Make sure aggregation has run for this dataset
    subprocess.check_call(['python', 'scripts/aggregate_results.py', '--dataset', DATASET])
    # Stage only the small artifacts (per-run JSONs are gitignored)
    subprocess.run(['git', 'add', f'results/{DATASET}_long.csv',
                                  f'results/{DATASET}_summary.csv',
                                  f'results/figures/'], check=False)
    diff = subprocess.run(['git', 'diff', '--cached', '--quiet']).returncode
    if diff == 0:
        print('[push] no new artifacts to commit')
    else:
        msg = f'E2 results: {DATASET} seed {SEED}'
        subprocess.check_call(['git', 'commit', '-m', msg])
        subprocess.check_call(['git', 'push', token_url, REPO_BRANCH])
        print(f'[push] pushed: {msg}')
else:
    print('[push] AUTO_PUSH=False; skipping')
